In [2]:
from lxml import etree
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter, defaultdict
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Configuration des chemins
CORPUS_DIR = "../../corpus"
VISUALIZATIONS_DIR = "visualisations"
TEI_NS = "http://www.tei-c.org/ns/1.0"
NSMAP = {'tei': TEI_NS}
NON_TRAITE = "NonTraite.txt"

# Créer les dossiers de sortie
Path(VISUALIZATIONS_DIR).mkdir(exist_ok=True)

print("✓ Configuration chargée")


✓ Configuration chargée


In [ ]:
def extract_named_entities_from_xml(xml_path):
    """Extrait les entités nommées (persName et placeName) d'un fichier XML."""
    parser = etree.XMLParser(remove_blank_text=False)
    tree = etree.parse(xml_path, parser)
    root = tree.getroot()
    
    entities = []
    
    # Extraire les persName
    pers_elements = root.xpath('.//tei:persName | .//persName', namespaces=NSMAP)
    for pers in pers_elements:
        text = ''.join(pers.itertext()).strip()
        ref = pers.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref if ref else text,
                'type': 'personne'
            })
    
    # Extraire les placeName
    place_elements = root.xpath('.//tei:placeName | .//placeName', namespaces=NSMAP)
    for place in place_elements:
        text = ''.join(place.itertext()).strip()
        ref = place.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref if ref else text,
                'type': 'lieu'
            })
    
    # Extraire les objectName
    place_elements = root.xpath('.//tei:objectName | .//objectName', namespaces=NSMAP)
    for place in place_elements:
        text = ''.join(place.itertext()).strip()
        ref = place.get('ref', '')
        if text:
            entities.append({
                'text': text,
                'ref': ref if ref else text,
                'type': 'oeuvre'
            })
    return entities


"""
CHARGEMENT DES DONNÉES
Charger les corpus français et italien
"""
def load_exclusion_list(filepath):
    """Charge la liste des morceaux de noms à exclure."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            # Lire les lignes et supprimer les espaces/retours à la ligne
            exclusions = [line.strip() for line in f if line.strip()]
        print(f"✓ Liste d'exclusion chargée : {len(exclusions)} entrées")
        return exclusions
    except FileNotFoundError:
        print(f"⚠ Fichier {filepath} non trouvé - aucune exclusion appliquée")
        return []

def should_exclude_file(filename, exclusion_list):
    """Vérifie si un fichier doit être exclu."""
    # Extraire la partie Auteur_Nom du nom de fichier
    # Ex: FRA_Auteur_Nom.xml -> Auteur_Nom
    name_without_ext = filename.stem  # Enlève l'extension
    parts = name_without_ext.split('_', 1)  # Split sur le premier underscore
    
    if len(parts) > 1:
        author_name = parts[0] 
        
        # Vérifier si un des morceaux d'exclusion est dans le nom
        for exclusion in exclusion_list:
            if exclusion in author_name:
                return True
    
    return False

def load_named_entities(directory, language, exclusion_list):
    """Charge toutes les entités nommées d'un corpus en excluant certains fichiers."""
    all_xml_files = list(Path(directory).glob(f'*.xml'))
    
    # Filtrer les fichiers à exclure
    xml_files = [f for f in all_xml_files if not should_exclude_file(f, exclusion_list)]
    
    all_entities = []
    for xml_file in xml_files:
        entities = extract_named_entities_from_xml(xml_file)
        all_entities.extend(entities)
    
    return pd.DataFrame(all_entities)

# CHARGER LA LISTE D'EXCLUSION
print("\n" + "="*60)
print("CHARGEMENT DE LA LISTE D'EXCLUSION")
print("="*60 + "\n")

exclusion_list = load_exclusion_list(NON_TRAITE)

# CHARGER LES DONNÉES
print("\n" + "="*60)
print("CHARGEMENT DES DONNÉES")
print("="*60 + "\n")

entities_PEINTURE = load_named_entities(CORPUS_DIR, 'PEINTURE', exclusion_list)
entities_ARCHI = load_named_entities(CORPUS_DIR, 'ARCHITECTURE', exclusion_list)
entities_PERSPECTIVE = load_named_entities(CORPUS_DIR, 'PERPECTIVE', exclusion_list)

print(f"\n✓ Données chargées:")
print(f"  - {len(entities_PEINTURE):,} entités")
print(f"  - {len(entities_ARCHI):,} entités")
print(f"  - {len(entities_PERSPECTIVE):,} entités")



CHARGEMENT DE LA LISTE D'EXCLUSION

✓ Liste d'exclusion chargée : 38 entrées

CHARGEMENT DES DONNÉES



XMLSyntaxError: xml:id : attribute value  is not an NCName, line 4, column 36 (../../corpus/IndexOeuvres.xml, line 4)